# 🎵 Advanced AI Music Studio
### 🛠️ Developed by: **Arafat Ahmed Mubin**
### 🌐 Brand: **2M**
---


### Enterprise-Grade Audio Generation & Stem Separation

This notebook provides a professional-tier audio production environment leveraging **Stable Audio Open 1.0** and **Meta's MusicGen**. 

**Core Capabilities:**
- **Text-to-Music:** High-fidelity audio up to 3 minutes (via chunked generation).
- **Audio Remixing:** Transform existing tracks using text prompts.
- **Stem Separation:** HTDemucs-powered split (Drums, Bass, Other, Vocals).
- **T4 Optimizations:** FP16, Sequential Offloading, and Active Cache Management.

## 🛠️ Step 1: Professional Environment Setup
We install the core audio stacks along with `demucs` for stem separation and `audiocraft` for melody-based remixing.

In [ ]:
# Install core audio stacks
!pip install -q -U diffusers transformers accelerate gradio soundfile torchaudio demucs audiocraft

## 🧠 Step 2: Advanced Audio Engine
We load the Stable Audio Open pipeline for base generation and configure HTDemucs for processing.

In [ ]:
import torch
import gc
import os
import numpy as np
import soundfile as sf
import gradio as gr
from diffusers import StableAudioPipeline
from audiocraft.models import MusicGen
import torchaudio
import subprocess

# Global Configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAO_REPO = "stabilityai/stable-audio-open-1.0"
MG_REPO = "facebook/musicgen-medium"

sao_pipe = None
mg_model = None

def clear_cache():
    gc.collect()
    torch.cuda.empty_cache()

def load_sao():
    global sao_pipe
    if sao_pipe is None:
        clear_cache()
        print("Loading Stable Audio Open...")
        sao_pipe = StableAudioPipeline.from_pretrained(SAO_REPO, torch_dtype=torch.float16)
        sao_pipe.enable_model_cpu_offload()
    return sao_pipe

def load_mg():
    global mg_model
    if mg_model is None:
        clear_cache()
        print("Loading MusicGen Remix Engine...")
        mg_model = MusicGen.get_pretrained(MG_REPO)
    return mg_model

def crossfade(chunk1, chunk2, sr, fade_len=2):
    """Seamlessly blends two audio chunks with crossfade."""
    fade_samples = int(fade_len * sr)
    fade_in = np.linspace(0, 1, fade_samples)
    fade_out = np.linspace(1, 0, fade_samples)
    
    chunk1[-fade_samples:] *= fade_out
    chunk2[:fade_samples] *= fade_in
    
    overlap = chunk1[-fade_samples:] + chunk2[:fade_samples]
    return np.concatenate([chunk1[:-fade_samples], overlap, chunk2[fade_samples:]])

## 🎛️ Step 3: Studio Core Functions
Implementation of 3-minute chunked generation, remixing, and stem splitting.

In [ ]:
def studio_generate(prompt, duration, loop):
    pipe = load_sao()
    sr = pipe.vae.sampling_rate
    
    # SAO generates up to 47s. We chunk for longer requests.
    num_chunks = int(np.ceil(duration / 40))
    final_audio = None
    
    for i in range(num_chunks):
        print(f"Generating chunk {i+1}/{num_chunks}...")
        chunk_duration = min(47, duration - (i * 40))
        out = pipe(prompt, num_inference_steps=100, audio_end_in_s=chunk_duration).audios[0]
        chunk = out.T.float().cpu().numpy().flatten()
        
        if final_audio is None:
            final_audio = chunk
        else:
            final_audio = crossfade(final_audio, chunk, sr)
    
    if loop:
        final_audio = np.concatenate([final_audio, final_audio])
        
    output_path = "studio_output.wav"
    sf.write(output_path, final_audio, sr)
    return output_path

def studio_remix(audio_input, prompt):
    model = load_mg()
    model.set_generation_params(duration=30)
    
    # Load melody for conditioning
    melody, sr = torchaudio.load(audio_input)
    out = model.generate_with_chroma([prompt], melody[None].to(DEVICE), sr)
    
    output_path = "remix_output.wav"
    torchaudio.save(output_path, out[0].cpu(), model.sample_rate)
    return output_path

def studio_split(audio_input):
    # Using Demucs CLI for high-quality HTDemucs separation
    cmd = ["python3", "-m", "demucs.separate", "-n", "htdemucs", audio_input, "--out", "stems"]
    subprocess.run(cmd)
    
    # Locate stems (usually in stems/htdemucs/filename/)
    base_name = os.path.basename(audio_input).split('.')[0]
    stem_dir = f"stems/htdemucs/{base_name}"
    
    return [
        f"{stem_dir}/drums.wav",
        f"{stem_dir}/bass.wav",
        f"{stem_dir}/other.wav",
        f"{stem_dir}/vocals.wav"
    ]

## 🏗️ Step 4: Multi-Tab Studio Interface
The futuristic Gradio dashboard for full control over the AI Music Studio.

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as studio:
    gr.Markdown("# 🎹 Advanced AI Music Studio")
    
    with gr.Tab("🚀 Generate"):
        with gr.Row():
            with gr.Column():
                gen_prompt = gr.Textbox(label="Musical Prompt", placeholder="Lofi hip hop, chill beats, rainy mood, 90bpm...")
                gen_dur = gr.Slider(5, 180, value=30, step=5, label="Duration (Seconds)")
                gen_loop = gr.Checkbox(label="Seamless Looping")
                gen_btn = gr.Button("Generate Masterpiece", variant="primary")
            with gr.Column():
                gen_out = gr.Audio(label="Output Audio", type="filepath")
                
    with gr.Tab("🌀 Remix"):
        with gr.Row():
            with gr.Column():
                remix_in = gr.Audio(label="Reference Audio", type="filepath")
                remix_prompt = gr.Textbox(label="Style Prompt", placeholder="Transform this into a heavy metal riff...")
                remix_btn = gr.Button("Remix Track", variant="primary")
            with gr.Column():
                remix_out = gr.Audio(label="Remixed Output", type="filepath")
                
    with gr.Tab("✂️ Stem Splitter"):
        with gr.Row():
            with gr.Column():
                split_in = gr.Audio(label="Track to Split", type="filepath")
                split_btn = gr.Button("Extract Stems", variant="primary")
            with gr.Column():
                stems_out = [gr.Audio(label=f"Stem: {s}", type="filepath") for s in ["Drums", "Bass", "Melody/Other", "Vocals/FX"]]
    
    gen_btn.click(studio_generate, [gen_prompt, gen_dur, gen_loop], gen_out)
    remix_btn.click(studio_remix, [remix_in, remix_prompt], remix_out)
    split_btn.click(studio_split, [split_in], stems_out)

studio.launch(share=True, debug=True)

---
**© 2026 2M | All Rights Reserved**
*Powered by the 2M Ecosystem (2M Dev's, 2M Future Facts, 2M Business Blogs)*